# LAB: Chaos Agent: Tools, Lore, Questionable Judgment

## Objective

Build a **creative complaint handler** for *NormalObjects* with a LangChain agent.
The agent picks its own tools, so you must **prove** the answer comes from tool output.

```
complaint → agent (LLM) → picks tools (any order) → tool outputs → final answer
```

## Steps

| # | Step | Code |
|---|------|---|
| 1 | Setup | PROVIDED |
| 2 | Create creative tools | PROVIDED + YOUR TURN |
| 3 | Create the agent | YOUR TURN |
| 4 | Test with sample complaints | YOUR TURN |
| 5 | Analyze agent behavior | YOUR TURN |
| | `lab_proof.md` | YOUR TURN |

**Submit:** `lab_proof.md` with code, one complaint, tool evidence, final output, one failure.

| Label | Meaning |
|---|---|
| **PROVIDED** | Run and read it |
| **YOUR TURN** | Replace the `# TODO` lines |
| **OPTIONAL** | Not required |

**Reference:** `langchain_integration.ipynb` (tools + `create_agent`)

**Prompt discipline:** use AI to review or debug, not to write your answer.

---
## Step 1: Setup [PROVIDED]

Add `OPENAI_API_KEY=...` to a `.env` file next to this notebook.

In [ ]:
# Install required packages (uncomment to run)
# !pip install -U langchain langchain-openai python-dotenv

In [1]:
import os
import random
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

model = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0.7)

OpenAI API Key loaded: True


**Done when:** the key prints `True` and there are no import errors.

---
## Step 2: Create Creative Tools

Every tool needs:
1. the `@tool` decorator
2. a clear docstring (the agent reads it to choose the tool)

### 2a. `consult_demogorgon` [PROVIDED]

In [2]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""
    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]
    return random.choice(responses)

In [9]:
print(consult_demogorgon.invoke({
    "complaint": "My lights keep flickering"
}))

The Demogorgon sniffs 'My lights keep flickering' and roars. It seems... hungry, not helpful.


### 2b. `check_hawkins_records` [YOUR TURN]

- Dictionary with 4 keys: `portal`, `monsters`, `psychics`, `electricity`
- Return the record whose key appears in the query (case-insensitive)
- No match → `"No record found ..."`

In [3]:
# YOUR TURN: Hawkins records tool
HAWKINS_RECORDS = {
    "portal": "A rift in Hawkins Lab opened a gateway to a dark parallel dimension.",
    "monsters": "Creatures from the other side hunt in the shadows around Hawkins.",
    "psychics": "Children with telekinetic powers were trained in secret experiments at the lab.",
    "electricity": "Lights flicker and power surges whenever the gate between worlds is active.",
}

@tool
def check_hawkins_records(query: str) -> str:
    """Search fictional Hawkins records by topic and return the first match."""
    query_lower = query.lower()
    for topic, record in HAWKINS_RECORDS.items():
        if topic in query_lower:
            return record
    return "No record found for that topic."

In [4]:
print(check_hawkins_records.invoke({"query": "Tell me about the PORTAL"}))
print(check_hawkins_records.invoke({"query": "My neighbor's cat is acting weird"}))

A rift in Hawkins Lab opened a gateway to a dark parallel dimension.
No record found for that topic.


### 2c. `cast_interdimensional_spell` [YOUR TURN]

- Args: `problem`, `creativity_level="medium"`
- Map `low/medium/high` → `1/2/3`
- 4 spell strings that include the problem
- `random.sample()` N spells, join with `"\n"`

In [5]:
# YOUR TURN: spell tool

@tool
def cast_interdimensional_spell(
    problem: str, creativity_level: str = "medium"
) -> str:
    """Suggest random fictional spells for a problem, not verified solutions."""
    multipliers = {"low": 1, "medium": 2, "high": 3}
    spells = [
        f"Draw a magical chalk circle around this problem: {problem}.",
        f"Channel the energy of the void to solve this issue: {problem}.",
        f"Invoke the spirits of the other side to assist with: {problem}.",
        f"Use the power of dimensional rifts to address: {problem}.",
    ]
    count = multipliers.get(creativity_level.lower(), 2)
    selected_spells = random.sample(spells, k=min(count, len(spells)))
    return "\n".join(selected_spells)

In [6]:
print(cast_interdimensional_spell.invoke({
    "problem": "Demodogs in my garden",
    "creativity_level": "high"
}))

Draw a magical chalk circle around this problem: Demodogs in my garden.
Channel the energy of the void to solve this issue: Demodogs in my garden.
Use the power of dimensional rifts to address: Demodogs in my garden.


### 2d. `gather_party_wisdom` [YOUR TURN]

- Dictionary with 4 keys, one per party member (Mike, Dustin, Lucas, Will)
- Return the match (case-insensitive)
- No match → generic "the party huddles..." answer

In [7]:
# YOUR TURN: party wisdom tool

@tool
def gather_party_wisdom(question: str) -> str:
    """Return fictional advice from a party member, or a generic fallback."""
    wisdom = {
        "mike": "Mike says: 'Always stick together, even when the lights go out.'",
        "dustin": "Dustin says: 'Trust your instincts, they know the way.'",
        "lucas": "Lucas says: 'Stay curious and keep exploring.'",
        "will": "Will says: 'Believe in yourself and your abilities.'",
    }
    question_lower = question.lower()
    for member, advice in wisdom.items():
        if member in question_lower:
            return advice
    return "The party huddles together to discuss the question."

In [8]:
print(gather_party_wisdom.invoke({"question": "What does DUSTIN suggest?"}))
print(gather_party_wisdom.invoke({"question": "What is the plan?"}))

Dustin says: 'Trust your instincts, they know the way.'
The party huddles together to discuss the question.


In [ ]:
# YOUR TURN: test each tool on its own with .invoke({...})

# TODO: print(consult_demogorgon.invoke({"complaint": "My lights keep flickering"}))
# TODO: print(check_hawkins_records.invoke({"query": "tell me about the portal"}))
# TODO: print(cast_interdimensional_spell.invoke({"problem": "Demodogs in my garden", "creativity_level": "high"}))
# TODO: print(gather_party_wisdom.invoke({"question": "what is the plan?"}))

**Done when:** all 4 tools return text, and a query with no matching key returns your fallback.

---
## Step 3: Create the Agent [YOUR TURN]

System prompt must:
- encourage creative tool use
- force at least one tool call
- say: answer **only** from tool results
- ask the agent to list the tools it used

**Hint:** `create_agent(model=..., tools=..., system_prompt=...)` in `langchain_integration.ipynb`.

In [ ]:
# YOUR TURN: tool list + system prompt + agent

tools = [
    consult_demogorgon,
    check_hawkins_records,
    cast_interdimensional_spell,
    gather_party_wisdom,
]

system_prompt = """
You are the NormalObjects Creative Complaint Handler.
Help users with fictional complaints about Hawkins and the Upside Down.

Use at least one tool before answering. Choose tools based on the complaint.
You may combine tools creatively when useful.

Base your answer only on the tool results from the current conversation.
Do not add facts from your own knowledge or invent missing evidence.
Do not treat the user's suspicions as confirmed facts.

Distinguish between:
- Hawkins records: information from our fictional source.
- Demogorgon responses: unreliable opinions.
- Party wisdom: fictional advice.
- Spells: imaginary suggestions, not verified solutions.

If the tool results do not support a conclusion, clearly say that there
is not enough evidence. A related record does not prove the cause of
the user's specific problem.

At the end, list the tools you actually used.
"""

# TODO: agent = create_agent(model=model, tools=tools, system_prompt=system_prompt)
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
)

print([t.name for t in tools])

['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']


**Done when:** the agent is created and prints the 4 tool names.

---
## Step 4: Test with Sample Complaints [YOUR TURN]

Use `config={"recursion_limit": 10}` to stop loops.

In [11]:
complaints = [
    "My Christmas lights keep flickering and I hear noises from the wall. What is going on?",
    "There is a weird portal in my basement. What's the plan?",
    "A Demodog ate my homework. I need a creative solution, creativity level high!",
    "My neighbor's cat is acting weird. Is it from the Upside Down?",  # no record exists
]

# YOUR TURN: run every complaint, keep the results for Step 5

results = []

for complaint in complaints:
    result = agent.invoke(
        {"messages": [HumanMessage(content=complaint)]},
        config={"recursion_limit": 10},
    )
    results.append(result)

    print("COMPLAINT:", complaint)
    print("FINAL ANSWER:", result["messages"][-1].content)
    print("-" * 60)

COMPLAINT: My Christmas lights keep flickering and I hear noises from the wall. What is going on?
FINAL ANSWER: There is no record in Hawkins about flickering Christmas lights or noises coming from the walls. The Demogorgon suggests simply turning off the lights, but this is not a reliable opinion.

Unfortunately, there isn't enough evidence to determine the specific cause of your issue with the lights and noises. It might be worth investigating further to ensure everything is safe.

Tools used: check_hawkins_records, consult_demogorgon.
------------------------------------------------------------
COMPLAINT: There is a weird portal in my basement. What's the plan?
FINAL ANSWER: According to Hawkins records, there have been instances where a rift opened a gateway to a dark parallel dimension. This suggests that your weird portal may be connected to similar phenomena.

As for what to do with it, the party wisdom is mixed, but here are some general suggestions: 

1. **Avoid it**: Stay cle

**Check:** for the cat complaint, does the agent admit "no record", or invent one?

---
## Step 5: Analyze Agent Behavior [YOUR TURN]

### 5a. Track tool usage

For each result: print the tool order (`a → b → c`). Then count total uses per tool.

**Hint:** AI messages have `.tool_calls`; use `collections.Counter`.

In [13]:
# YOUR TURN: tool order + counts

from collections import Counter

counts = Counter()

for complaint, result in zip(complaints, results):
    order = [
        call["name"]
        for msg in result["messages"]
        if msg.type == "ai"
        for call in msg.tool_calls
    ]
    counts.update(order)

    print("COMPLAINT:", complaint)
    print(" → ".join(order) if order else "No tools used")
    print()

print("TOTAL USES:", dict(counts))

COMPLAINT: My Christmas lights keep flickering and I hear noises from the wall. What is going on?
check_hawkins_records → consult_demogorgon

COMPLAINT: There is a weird portal in my basement. What's the plan?
check_hawkins_records → gather_party_wisdom

COMPLAINT: A Demodog ate my homework. I need a creative solution, creativity level high!
cast_interdimensional_spell

COMPLAINT: My neighbor's cat is acting weird. Is it from the Upside Down?
consult_demogorgon → gather_party_wisdom

TOTAL USES: {'check_hawkins_records': 2, 'consult_demogorgon': 2, 'gather_party_wisdom': 2, 'cast_interdimensional_spell': 1}


### 5b. Trace one answer (grounding)

For one result, print every tool call, its args, and what it returned.

**Hint:** messages with `msg.type == "tool"` hold the tool output.

In [12]:
# YOUR TURN: evidence trace for one result

for msg in results[1]["messages"]:
    if msg.type == "ai":
        for call in msg.tool_calls:
            print("TOOL CALL:", call["name"])
            print("ARGS:", call["args"])
            print("CALL ID:", call["id"])

    elif msg.type == "tool":
        print("TOOL OUTPUT:", msg.name)
        print("CALL ID:", msg.tool_call_id)
        print("CONTENT:", msg.content)
        print("-" * 60)

print("FINAL ANSWER:")
print(results[1]["messages"][-1].content)

TOOL CALL: check_hawkins_records
ARGS: {'query': 'portal in basement'}
CALL ID: call_4lllqoxO7oiL4xKpFTZGyDP0
TOOL CALL: gather_party_wisdom
ARGS: {'question': 'What to do with a weird portal?'}
CALL ID: call_FC07AqIjllhjHgWFozOcXIZ0
TOOL OUTPUT: check_hawkins_records
CALL ID: call_4lllqoxO7oiL4xKpFTZGyDP0
CONTENT: A rift in Hawkins Lab opened a gateway to a dark parallel dimension.
------------------------------------------------------------
TOOL OUTPUT: gather_party_wisdom
CALL ID: call_FC07AqIjllhjHgWFozOcXIZ0
CONTENT: The party huddles together to discuss the question.
------------------------------------------------------------
FINAL ANSWER:
According to Hawkins records, there have been instances where a rift opened a gateway to a dark parallel dimension. This suggests that your weird portal may be connected to similar phenomena.

As for what to do with it, the party wisdom is mixed, but here are some general suggestions: 

1. **Avoid it**: Stay clear of the portal until you gathe

**Check:** mark each claim in the final answer as either from a tool or invented.

### 5c. Compare with a structured approach

Write `structured_handler(complaint)`: always calls records → party → spell (`low`), no LLM.

In [14]:
# YOUR TURN: fixed-order handler

def structured_handler(complaint: str) -> str:
    # 1. Llama a check_hawkins_records con query=complaint.
    #    Guarda el resultado en records_result.
    records_result = check_hawkins_records.invoke({"query": complaint})

    # 2. Llama a gather_party_wisdom con question=complaint.
    #    Guarda el resultado en party_result.
    party_result = gather_party_wisdom.invoke({"question": complaint})

    # 3. Llama a cast_interdimensional_spell con:
    #    problem=complaint y creativity_level="low".
    #    Guarda el resultado en spell_result.
    spell_result = cast_interdimensional_spell.invoke({"problem": complaint, "creativity_level": "low"})

    return (
        f"RECORDS: {records_result}\n"
        f"PARTY ADVICE: {party_result}\n"
        f"FICTIONAL SPELL: {spell_result}"
    )
# TODO: print(structured_handler(complaints[1]))
print(structured_handler("There is a weird portal in my basement. What's the plan?"))


RECORDS: A rift in Hawkins Lab opened a gateway to a dark parallel dimension.
PARTY ADVICE: The party huddles together to discuss the question.
FICTIONAL SPELL: Channel the energy of the void to solve this issue: There is a weird portal in my basement. What's the plan?.


| | Agent | Structured |
|---|---|---|
| Same tools every run? |eligió herramientas diferentes según la queja en las pruebas realizadas| el flujo fijo siempre llama a las mismas tres herramientas.|
| Tool order predictable? |decide qué llamar; no tiene ese orden obligatorio. | registros → grupo → hechizo|
| Can invent facts? |añadió consejos sin respaldo | devuelve los textos de las herramientas sin ampliarlos, aunque esos textos contienen ficción. |
| Easy to trace? |necesitamos inspeccionar sus llamadas, argumentos y resultados |es más directo de revisar porque tiene tres llamadas explícitas |
| Best for which client case? |Consultas variadas que requieren elegir herramientas según la necesidad del cliente |Procesos que deben seguir siempre pasos obligatorios en un orden definido |

---
## Extensions [OPTIONAL]

- More tools (`consult_eleven`, `check_government_files`)
- LLM call inside a tool
- Memory with `InMemorySaver` + `thread_id`
- Streamlit / Gradio UI
- Measure response time and tool calls per complaint

In [ ]:
# YOUR TURN (optional)

# TODO

---
## Write `lab_proof.md` [YOUR TURN]

```markdown
# Lab Proof: NormalObjects Creative Complaint Handler

## 1. What I built
## 2. Query
## 3. Tool evidence (calls + args + outputs)
## 4. Final output
## 5. Trace (which claim comes from which tool output?)
## 6. Failure or limitation (where can it give a plausible but unsupported answer?)
## 7. Agent vs. structured handler
```

**Before you submit:**
- [ ] No API keys or `.env`
- [ ] Only lab files
- [ ] `lab_proof.md` is the entry point

---
### Troubleshooting

| Problem | Fix |
|---|---|
| Agent doesn't use tools | Clearer docstrings; prompt must require tool use |
| Tools in "wrong" order | Expected for agents. Need fixed order? Use LangGraph |
| Agent loops | Lower `recursion_limit`, improve docstrings |
| Generic answers | More specific tool outputs, lower temperature |
| `ImportError` on `create_agent` | `pip install -U langchain` (needs v1+) |